In [5]:
import pandas as pd
import numpy as np
rng = np.random.default_rng(42)
n = 2000
raw = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "user_id":  rng.integers(1, 200, n),
    "city":     rng.choice(["北京", "上海", "广州", "深圳", "成都", ""], n, p=[.2,.2,.15,.15,.15,.15]),
    "category": rng.choice(["数码", "服饰", "食品", "家居", "图书"], n),
    "amount":   np.round(rng.lognormal(4, 0.8), 2),
    "qty":      rng.integers(1, 6, n),
    "pay_time": pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 90*24*3600, n), unit="s"),
    "status":   rng.choice(["paid", "paid", "paid", "refunded", "cancelled"], n),
})
#raw.loc(rng.choice(raw.index,80),"amount") = -1
raw.loc[rng.choice(raw.index,40),"pay_time"] = None
raw.loc[rng.choice(raw.index,30),"city"] = " 北 京 "
raw.loc[rng.choice(raw.index,20),"user_id"] = -1
raw.loc[rng.choice(raw.index,15),"qty"] = 0
raw = raw.sample(frac=1).reset_index(drop=True)
print(raw.info());print(raw.describe(include="all"));print(raw.isna().mean())

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   order_id  2000 non-null   int64         
 1   user_id   2000 non-null   int64         
 2   city      2000 non-null   str           
 3   category  2000 non-null   str           
 4   amount    2000 non-null   float64       
 5   qty       2000 non-null   int64         
 6   pay_time  1961 non-null   datetime64[us]
 7   status    2000 non-null   str           
dtypes: datetime64[us](1), float64(1), int64(3), str(3)
memory usage: 125.1 KB
None
           order_id      user_id  city category   amount          qty  \
count   2000.000000  2000.000000  2000     2000  2000.00  2000.000000   
unique          NaN          NaN     7        5      NaN          NaN   
top             NaN          NaN    北京       数码      NaN          NaN   
freq            NaN          NaN   393      425      NaN          NaN

In [8]:
df = raw.copy()
df["city"] = df["city"].str.strip()
df.loc[df["city"]=="","city"] = "未知"
df["amount"] = pd.to_numeric(df["amount"],errors="coerce")
df = df[(df["amount"]>0)&(df["qty"]>0)&(df["user_id"]>0)].reset_index(drop=True)
df = df.dropna(subset=["pay_time"])
df["month"] = df["pay_time"].dt.to_period("M")
df["unit_price"] = df["amount"] / df["qty"]
gmv = (df.groupby("city")
 .agg(订单数=("order_id", "count"),
      GMV=("amount", "sum"),
      客单价=("amount", "mean"),
      退款率=("status", lambda s: (s == "refunded").mean()))
 .sort_values("GMV", ascending=False)
 .assign(GMV=lambda d: d["GMV"].round(0), 客单价=lambda d: d["客单价"].round(2)))
print(gmv)

      订单数     GMV    客单价       退款率
city                              
北京    378  8978.0  23.75  0.203704
上海    377  8954.0  23.75  0.175066
广州    302  7172.0  23.75  0.205298
未知    292  6935.0  23.75  0.229452
成都    277  6579.0  23.75  0.194946
深圳    274  6508.0  23.75  0.200730
北 京    30   712.0  23.75  0.100000


In [12]:
trend = (df.groupby("month")["amount"].sum()
         .rename("GMV").reset_index().assign(month=lambda d: d["month"].astype(str)))
trend["环比"] = trend["GMV"].pct_change().round(4)
print(trend)

     month       GMV      环比
0  2024-01  15722.50     NaN
1  2024-02  14986.25 -0.0468
2  2024-03  15128.75  0.0095


In [20]:
snapshot = pd.Timestamp("2024-04-01")
rfm = (df[df["status"] == "paid"]
 .groupby("user_id")
 .agg(R=("pay_time", lambda s: (snapshot - s.max()).days),
      F=("order_id", "count"),
      M=("amount", "sum"))
 .reset_index())
for col in ['R','F','M']:
    rfm[f'{col}_score'] = pd.qcut(rfm[col],q=4,labels=[4,3,2,1] if col=='R' else [1,2,3,4],duplicates='drop').astype(int)
rfm['RFM'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)
print(rfm.sort_values('M',ascending=False).head(10))

     user_id   R   F       M  R_score  F_score  M_score  RFM
155      156  17  14  332.50        2        4        4  244
8          9   3  13  308.75        4        4        4  444
23        24   3  11  261.25        4        4        4  444
105      106   6  11  261.25        3        4        4  344
146      147  16  11  261.25        2        4        4  244
126      127   2  11  261.25        4        4        4  444
60        61  16  11  261.25        2        4        4  244
191      192   7  11  261.25        3        4        4  344
100      101   3  10  237.50        4        4        4  444
102      103   6  10  237.50        3        4        4  344
